# Rental Rights Knowledge Base Preprocessing

This notebook prepares the source documents used in the Victorian Rental Rights RAG knowledge base. It extracts, cleans and standardises the documents before creating the retrieval-ready dataset.

## 1. Load Source Documents

Set the project paths and confirm the initial rental rights PDFs are available.

In [3]:
from pathlib import Path
import pymupdf
import pandas as pd

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "raw").exists():
    DATA_DIR = CURRENT_DIR / "data" / "raw"
elif (CURRENT_DIR.parent / "data" / "raw").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "raw"
else:
    raise FileNotFoundError("Could not locate data/raw folder")

renters_guide_path = DATA_DIR / "Renters Guide.pdf"
minimum_standards_path = DATA_DIR / "Rental minimum standards checklist for renters PDF.pdf"

print("Using data from:", DATA_DIR)
print("Renters Guide found:", renters_guide_path.exists())
print("Minimum Standards found:", minimum_standards_path.exists())

Using data from: /Users/seanrichards/Documents/University/DS Case Studies/Project/data/raw
Renters Guide found: True
Minimum Standards found: True


## 2. Extract PDF Text

Extract each PDF page while keeping its document and page metadata for later source tracking.

In [4]:
def extract_pdf_pages(file_path, document_id, document_name):
    rows = []

    pdf = pymupdf.open(file_path)

    for page_num, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        rows.append({
            "document_id": document_id,
            "document_name": document_name,
            "page": page_num,
            "text": text,
            "char_count": len(text)
        })

    pdf.close()
    return rows


pages = []

pages += extract_pdf_pages(
    renters_guide_path,
    "RG",
    "Renters Guide"
)

pages += extract_pdf_pages(
    minimum_standards_path,
    "MS",
    "Rental Minimum Standards Checklist"
)

pages_df = pd.DataFrame(pages)

print("Total pages extracted:", len(pages_df))
display(pages_df.head())

Total pages extracted: 42


,document_id,document_name,page,text,char_count
0,RG,Renters Guide,1,Renters \nGuide\nconsumer.vic.gov.au,34
1,RG,Renters Guide,2,"Disclaimer, copyright and publisher informatio...",1547
2,RG,Renters Guide,3,Introduction \t\n4\nFinding and applying for a...,439
3,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...,1369
4,RG,Renters Guide,5,Where to find more information\nVisit our webs...,503


## 3. Check Extraction Quality

Inspect a few representative pages to make sure the useful text, headings and lists were extracted correctly.

In [ ]:
#Inspect a few useful pages from both documents
pages_to_check = [
    ("RG", 13),   # Minimum standards / heating
    ("RG", 24),   # Repairs
    ("RG", 28),   # Inspections / entry
    ("MS", 1),    # Minimum standards checklist page 1
    ("MS", 2)     # Minimum standards checklist page 2
]

for document_id, page_num in pages_to_check:
    row = pages_df[
        (pages_df["document_id"] == document_id) &
        (pages_df["page"] == page_num)
    ].iloc[0]

    print("=" * 80)
    print(f"{row['document_name']} - Page {page_num}")
    print("=" * 80)
    print(row["text"])
    print()

Renters Guide - Page 13
Learn more about each of the  
minimum standards below.
For a complete list of the standards and  
possible exemptions, scan the QR code or visit:  
consumer.vic.gov.au/rentalstandards
	
Bathrooms
A rental property’s bathroom must have a washbasin and a shower or bath, 
and be connected to a reasonable supply of hot and cold water. 
Showers must have a shower head with a 3-star water efficiency rating.  
If a 3-star shower head can’t be installed, for example because of the 
property’s age, then a shower head with a 1- or 2-star rating is acceptable.
	
Electrical safety
Rental properties must have modern switchboards, with circuit breakers  
and electrical safety switches installed. Electrical safety switches are known  
as residual current devices (RCD, RCCB or RCBO).
Rental providers are responsible for engaging an electrician to ensure their 
rental property complies with the electrical safety standard.
	
Heating
All rental properties must have a fixed heater

## 4. Clean Extracted Text

Remove basic PDF formatting noise while keeping the document structure and content intact.

In [6]:
import re

def clean_page_text(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Remove obvious PDF footer noise
        if re.fullmatch(r"\d+", line):
            continue
        if line == "Renters Guide":
            continue

        # Clean tabs and repeated spaces
        line = re.sub(r"\t+", " ", line)
        line = re.sub(r" {2,}", " ", line)

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # Join words split across PDF line breaks
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


pages_df["clean_text"] = pages_df["text"].apply(clean_page_text)

# Compare one page before and after
example = pages_df[
    (pages_df["document_id"] == "RG") &
    (pages_df["page"] == 24)
].iloc[0]

print("RAW TEXT:\n")
print(example["text"])

print("\n" + "=" * 80)
print("CLEANED TEXT:\n")
print(example["clean_text"])

RAW TEXT:

Problems with the property 
Your rental provider must make sure the property is in good condition and fit to 
live in. It doesn’t matter how much rent you are paying or how old the property is.
If there’s a problem with the property, you can ask your rental provider to fix it. 
If they don’t, contact Consumer Affairs Victoria for information and advice.
Repairs
Repairs are either ‘urgent’ or ‘non-urgent’. Rental providers must make  
urgent repairs immediately. Rental providers must make non-urgent repairs 
within 14 days of getting a written request.
Urgent repairs
Anything on this list is legally defined as an urgent repair:
•	 burst water service
•	 blocked or broken toilet system
•	 serious roof leak
•	 gas leak
•	 dangerous electrical fault
•	 flooding or serious flood damage
•	 serious storm or fire damage
•	 an essential service or appliance for hot water, water, cooking, heating,  
or laundering isn’t working
•	 the gas, electricity or water supply isn’t working
•	 a

## 5. Create Baseline Chunks

Use one PDF page as one chunk and assign a unique ID so retrieved information can be traced back to its source.

In [ ]:
#Create one baseline knowledge-base chunk per PDF page
kb_df = pages_df.copy()

kb_df["chunk_id"] = (
    kb_df["document_id"] +
    "_P" +
    kb_df["page"].astype(str).str.zfill(2)
)

kb_df = kb_df[
    ["chunk_id", "document_id", "document_name", "page", "clean_text"]
].rename(columns={"clean_text": "text"})

print("Total knowledge-base chunks:", len(kb_df))
display(kb_df.head(10))

Total knowledge-base chunks: 42


,chunk_id,document_id,document_name,page,text
0,RG_P01,RG,Renters Guide,1,Renters\nGuide\nconsumer.vic.gov.au
1,RG_P02,RG,Renters Guide,2,"Disclaimer, copyright and publisher informatio..."
2,RG_P03,RG,Renters Guide,3,Introduction\nFinding and applying for a renta...
3,RG_P04,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...
4,RG_P05,RG,Renters Guide,5,Where to find more information\nVisit our webs...
5,RG_P06,RG,Renters Guide,6,Finding\nand applying\nfor a rental property
6,RG_P07,RG,Renters Guide,7,Before you apply\nDocuments and information yo...
7,RG_P08,RG,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...
8,RG_P09,RG,Renters Guide,9,Read through and complete the rental applicati...
9,RG_P10,RG,Renters Guide,10,Before you move\ninto a rental\nproperty


## 6. Review Low-Value Chunks

Check the shortest chunks for cover pages, contents pages and section dividers that are unlikely to help retrieval.

In [8]:
# Inspect the shortest chunks in the knowledge base

chunk_lengths = kb_df.copy()
chunk_lengths["char_count"] = chunk_lengths["text"].str.len()

display(
    chunk_lengths[
        ["chunk_id", "document_name", "page", "char_count", "text"]
    ]
    .sort_values("char_count")
    .head(12)
)

,chunk_id,document_name,page,char_count,text
36,RG_P37,Renters Guide,37,29,If you’re in\na rental dispute
29,RG_P30,Renters Guide,30,31,Moving out of a\nrental property
0,RG_P01,Renters Guide,1,33,Renters\nGuide\nconsumer.vic.gov.au
19,RG_P20,Renters Guide,20,37,After you\nmove into a\nrental property
9,RG_P10,Renters Guide,10,38,Before you move\ninto a rental\nproperty
5,RG_P06,Renters Guide,6,42,Finding\nand applying\nfor a rental property
39,RG_P40,Renters Guide,40,130,Need help in your language?\nCall TIS National...
2,RG_P03,Renters Guide,3,379,Introduction\nFinding and applying for a renta...
10,RG_P11,Renters Guide,11,453,Communicating with your rental provider\nYou c...
4,RG_P05,Renters Guide,5,482,Where to find more information\nVisit our webs...


## 7. Build the Retrieval Knowledge Base

Remove non-content pages from the retrieval set while keeping the original extracted data unchanged.

In [9]:
# Remove pages that do not contain useful rental-rights content

exclude_chunks = [
    "RG_P01",
    "RG_P02",
    "RG_P03",
    "RG_P05",
    "RG_P06",
    "RG_P10",
    "RG_P20",
    "RG_P30",
    "RG_P37",
    "RG_P40"
]

kb_retrieval_df = kb_df[
    ~kb_df["chunk_id"].isin(exclude_chunks)
].reset_index(drop=True)

print("Original chunks:", len(kb_df))
print("Chunks used for retrieval:", len(kb_retrieval_df))

display(kb_retrieval_df.head())

Original chunks: 42
Chunks used for retrieval: 32


,chunk_id,document_id,document_name,page,text
0,RG_P04,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...
1,RG_P07,RG,Renters Guide,7,Before you apply\nDocuments and information yo...
2,RG_P08,RG,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...
3,RG_P09,RG,Renters Guide,9,Read through and complete the rental applicati...
4,RG_P11,RG,Renters Guide,11,Communicating with your rental provider\nYou c...


## 8. Save the Processed Knowledge Base

Save the final 32 retrieval chunks as JSONL for use in the RAG and evaluation notebooks.

In [10]:
# Save the processed Version 1 knowledge base

PROCESSED_DIR = DATA_DIR.parent / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

kb_retrieval_df["source_file"] = kb_retrieval_df["document_id"].map({
    "RG": "Renters Guide.pdf",
    "MS": "Rental minimum standards checklist for renters PDF.pdf"
})

output_path = PROCESSED_DIR / "rental_kb_chunks.jsonl"

kb_retrieval_df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved knowledge base to:")
print(output_path)

print("\nChunks saved:", len(kb_retrieval_df))

Saved knowledge base to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/processed/rental_kb_chunks.jsonl

Chunks saved: 32
